In [1]:
import os
import glob
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib

# -----------------------------
# FEATURE EXTRACTION
# -----------------------------
def extract_mfcc_features(audio_path, n_mfcc=13, n_fft=2048, hop_length=512):
    try:
        audio_data, sr = librosa.load(audio_path, sr=None)
    except Exception as e:
        print(f"Error loading audio file {audio_path}: {e}")
        return None

    mfccs = librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
    return np.mean(mfccs.T, axis=0)

# -----------------------------
# DATASET CREATION
# -----------------------------
def create_dataset(directory, label):
    X, y = [], []
    print(f"Checking directory: {directory}")
    audio_files = glob.glob(os.path.join(directory, "*.wav"))
    print(f"Found {len(audio_files)} audio files")

    for audio_path in audio_files:
        mfcc_features = extract_mfcc_features(audio_path)
        if mfcc_features is not None:
            X.append(mfcc_features)
            y.append(label)
        else:
            print(f"Skipping audio file {audio_path}")

    print("Number of valid samples in", directory, ":", len(X))
    return X, y

# -----------------------------
# MODEL TRAINING
# -----------------------------
def train_model(X, y):
    unique_classes = np.unique(y)
    print("Unique classes in dataset:", unique_classes)

    if len(unique_classes) < 2:
        raise ValueError("At least two different classes (real/fake) are required to train.")

    print("Shape of X:", X.shape)
    print("Shape of y:", y.shape)

    # Check for small class sizes
    class_counts = np.bincount(y)
    if np.min(class_counts) < 2:
        print("Not enough samples per class for stratified splitting. Training on all data.")
        X_train, y_train = X, y
        X_test, y_test = None, None
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        print("Train/Test split completed.")
        print("X_train:", X_train.shape)
        print("X_test:", X_test.shape)

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    svm_classifier = SVC(kernel='linear', random_state=42)

    if X_test is not None:
        X_test_scaled = scaler.transform(X_test)
        svm_classifier.fit(X_train_scaled, y_train)

        y_pred = svm_classifier.predict(X_test_scaled)

        accuracy = accuracy_score(y_test, y_pred)
        confusion_mtx = confusion_matrix(y_test, y_pred)

        print("Accuracy:", accuracy)
        print("Confusion Matrix:\n", confusion_mtx)
    else:
        svm_classifier.fit(X_train_scaled, y_train)
        print("Trained model without test split (insufficient samples).")

    # Save model and scaler
    joblib.dump(svm_classifier, "model.pkl")
    joblib.dump(scaler, "scaler.pkl")
    print("Model and scaler saved successfully.")

# -----------------------------
# MAIN FUNCTION
# -----------------------------
def main():
    # ✅ Updated dataset directories for Kaggle
    real_dir = "/kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/real"
    fake_dir = "/kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/fake"

    X_real, y_real = create_dataset(real_dir, label=0)
    X_fake, y_fake = create_dataset(fake_dir, label=1)

    if len(X_real) == 0 and len(X_fake) == 0:
        print("No audio files found in either directory.")
        return

    X = np.vstack((X_real, X_fake))
    y = np.hstack((y_real, y_fake))

    train_model(X, y)


if __name__ == "__main__":
    main()

Checking directory: /kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/real
Found 32 audio files
Number of valid samples in /kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/real : 32
Checking directory: /kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/fake
Found 31 audio files
Number of valid samples in /kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/fake : 31
Unique classes in dataset: [0 1]
Shape of X: (63, 13)
Shape of y: (63,)
Train/Test split completed.
X_train: (50, 13)
X_test: (13, 13)
Accuracy: 1.0
Confusion Matrix:
 [[7 0]
 [0 6]]
Model and scaler saved successfully.


In [2]:
import os
import glob
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib

# -----------------------------
# FEATURE EXTRACTION
# -----------------------------
def extract_mfcc_features(audio_path, n_mfcc=13, n_fft=2048, hop_length=512):
    try:
        audio_data, sr = librosa.load(audio_path, sr=None)
    except Exception as e:
        print(f"Error loading audio file {audio_path}: {e}")
        return None

    mfccs = librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
    return np.mean(mfccs.T, axis=0)

# -----------------------------
# DATASET CREATION
# -----------------------------
def create_dataset(directory, label):
    X, y = [], []
    print(f"Checking directory: {directory}")
    audio_files = glob.glob(os.path.join(directory, "*.wav"))
    print(f"Found {len(audio_files)} audio files")

    for audio_path in audio_files:
        mfcc_features = extract_mfcc_features(audio_path)
        if mfcc_features is not None:
            X.append(mfcc_features)
            y.append(label)
        else:
            print(f"Skipping audio file {audio_path}")

    print("Number of valid samples in", directory, ":", len(X))
    return X, y

# -----------------------------
# MODEL TRAINING
# -----------------------------
def train_model(X, y):
    unique_classes = np.unique(y)
    print("Unique classes in dataset:", unique_classes)

    if len(unique_classes) < 2:
        raise ValueError("At least two different classes (real/fake) are required to train.")

    print("Shape of X:", X.shape)
    print("Shape of y:", y.shape)

    # Check for small class sizes
    class_counts = np.bincount(y)
    if np.min(class_counts) < 2:
        print("Not enough samples per class for stratified splitting. Training on all data.")
        X_train, y_train = X, y
        X_test, y_test = None, None
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        print("Train/Test split completed.")
        print("X_train:", X_train.shape)
        print("X_test:", X_test.shape)

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    svm_classifier = SVC(kernel='linear', random_state=42)

    if X_test is not None:
        X_test_scaled = scaler.transform(X_test)
        svm_classifier.fit(X_train_scaled, y_train)

        y_pred = svm_classifier.predict(X_test_scaled)

        accuracy = accuracy_score(y_test, y_pred)
        confusion_mtx = confusion_matrix(y_test, y_pred)

        print("Accuracy:", accuracy)
        print("Confusion Matrix:\n", confusion_mtx)
    else:
        svm_classifier.fit(X_train_scaled, y_train)
        print("Trained model without test split (insufficient samples).")

    # Save model and scaler
    # Save model and scaler
    joblib.dump(svm_classifier, "/kaggle/working/model.pkl")
    joblib.dump(scaler, "/kaggle/working/scaler.pkl")
    print("Model and scaler saved successfully in /kaggle/working/")


# -----------------------------
# MAIN FUNCTION
# -----------------------------
def main():
    # ✅ Updated dataset directories for Kaggle
    real_dir = "/kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/real"
    fake_dir = "/kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/fake"

    X_real, y_real = create_dataset(real_dir, label=0)
    X_fake, y_fake = create_dataset(fake_dir, label=1)

    if len(X_real) == 0 or len(X_fake) == 0:
        print("No audio files found in either directory.")
        return

    X = np.vstack((X_real, X_fake))
    y = np.hstack((y_real, y_fake))

    train_model(X, y)


if __name__ == "__main__":
    main()

Checking directory: /kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/real
Found 32 audio files
Number of valid samples in /kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/real : 32
Checking directory: /kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/fake
Found 31 audio files
Number of valid samples in /kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/fake : 31
Unique classes in dataset: [0 1]
Shape of X: (63, 13)
Shape of y: (63,)
Train/Test split completed.
X_train: (50, 13)
X_test: (13, 13)
Accuracy: 1.0
Confusion Matrix:
 [[7 0]
 [0 6]]
Model and scaler saved successfully in /kaggle/working/


In [3]:
import os
import librosa
import numpy as np
import joblib

def extract_mfcc_features(audio_path, n_mfcc=13, n_fft=2048, hop_length=512):
    try:
        audio_data, sr = librosa.load(audio_path, sr=None)
    except Exception as e:
        print(f"Error loading audio file {audio_path}: {e}")
        return None

    mfccs = librosa.feature.mfcc(
        y=audio_data, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length
    )
    return np.mean(mfccs.T, axis=0)


def analyze_audio(input_audio_path):
    model_filename = "/kaggle/working/model.pkl"
    scaler_filename = "/kaggle/working/scaler.pkl"


    if not os.path.exists(model_filename) or not os.path.exists(scaler_filename):
        return "Error: Model or scaler file not found. Train the model first."

    svm_classifier = joblib.load(model_filename)
    scaler = joblib.load(scaler_filename)

    if not os.path.exists(input_audio_path):
        return "Error: The specified file does not exist."
    elif not input_audio_path.lower().endswith(".wav"):
        return "Error: The specified file is not a .wav file."

    mfcc_features = extract_mfcc_features(input_audio_path)

    if mfcc_features is not None:
        mfcc_features_scaled = scaler.transform(mfcc_features.reshape(1, -1))
        prediction = svm_classifier.predict(mfcc_features_scaled)

        # ❌ Don't delete files in Kaggle input directories
        # os.remove(input_audio_path)

        if prediction[0] == 0:
            return "The input audio is classified as real."
        else:
            return "The input audio is classified as fake."
    else:
        return "Error: Unable to process the input audio."


if __name__ == "__main__":
    user_input_file = input("Enter the path of the .wav file to analyze: ")
    result = analyze_audio(user_input_file)
    print(result)

Enter the path of the .wav file to analyze:  /kaggle/input/datasets/prasad1706/livedata11/dataspace/dataset/dataset/real/WhatsApp Audio 2026-02-15 at 11.56.55 PM.wav


The input audio is classified as real.
